# 14 — The richer embedding, applied

Notebook `13` left us with a sharp, structural diagnosis: on the **naive phase qubit**, the joint state $\rho_{AB}$ has a single coupling-dependent number in it, and that number is the phase-locking value. Feed a quantum measure a state that only knows PLV and it can only report PLV. The bottleneck was never quantum optimal transport — it was the embedding. This notebook builds the fix.

**Prerequisites:** `13_the_redundancy_reveal` (the redundancy is a property of the *naive embedding*, not of quantum OT), and `10_embedding_signals_into_states` (the embedding menu, where the multi-frequency qutrit was introduced).

**What you'll be able to do**
- Apply the **multi-frequency qutrit** $\tfrac{1}{\sqrt 3}\bigl(|0\rangle + e^{i\theta}|1\rangle + e^{2i\theta}|2\rangle\bigr)$ to the Kuramoto dyad and build its $9\times 9$ joint density $\rho_{AB}$.
- Locate the **two** coupling-bearing coherences: $\rho_{AB}[3,1]$ (the first circular moment — the naive embedding's only channel) and $\rho_{AB}[6,2]$ (the **second** moment — a channel the naive qubit did not even have).
- Show that $\rho_{AB}[6,2]$ carries real structure on the dyad: $|\rho_{AB}[6,2]|$ grows with the coupling $K$ as the dyad locks.
- See, with a short two-sample illustration, that the second moment is an **independent** degree of freedom — two couplings can share a PLV (matched $\rho_{AB}[3,1]$) yet differ in $\rho_{AB}[6,2]$ — so the redundancy of `13` is broken in principle.

This is the constructive half of the capstone. In `13` you proved *why* the naive measure could not beat PLV; here you enrich the embedding so $\rho_{AB}$ carries structure beyond PLV, opening a channel the phase-locking value cannot see. The tool you build today is what the discriminating experiment of `15` will turn into a measurable advantage.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum_ot.capstone import (
    simulate_kuramoto_dyad,
    joint_density_matrix,
    plv,
)
from qot_course.quantum_ot.embeddings import (
    multifreq_state,
    joint_density_from_states,
)
from qot_course.quantum_ot.discrimination import sample_phase_difference

np.random.seed(42)
viz.use_course_style()

## 1. The fix, stated

Here is the exact shape of the problem `13` handed us. The naive phase qubit maps each phase to $(|0\rangle + e^{i\theta}|1\rangle)/\sqrt 2$ — one phase, one equatorial qubit. When we form the joint state of two such qubits and time-average, the only matrix element that moves with the coupling is the off-block coherence linking $|01\rangle$ and $|10\rangle$, and its magnitude is exactly $\mathrm{PLV}/4$. The diagonal is frozen at $\tfrac14 I_4$; the marginals sit at $I/2$. So $\rho_{AB}$ has one coupling-dependent degree of freedom, and it is the first circular moment of the phase difference, $\langle e^{i\Delta\theta}\rangle$ — which is what PLV reports. A measure reading that state has nothing else to read.

The diagnosis points straight at the cure. The redundancy was a property of *how thin the embedding was*, not of the quantum measure. If we want $\rho_{AB}$ to carry structure beyond PLV, we must hand it a richer state — one whose coherences encode *more* than the first moment.

That richer state is already on the menu from `10`: the **multi-frequency qutrit**

$$ |\psi(\theta)\rangle \;=\; \frac{1}{\sqrt 3}\bigl(\,|0\rangle + e^{i\theta}|1\rangle + e^{2i\theta}|2\rangle\,\bigr). $$

The naive qubit had two levels and could only hold $e^{i\theta}$. This qutrit adds a third level carrying $e^{2i\theta}$ — the phase's *second harmonic*. When two qutrits are joined, the joint density gains a second coupling-bearing coherence sitting at the second circular moment $\langle e^{2i\Delta\theta}\rangle$, in a matrix element the naive embedding did not possess. Circular moments are the standard descriptors of a phase distribution (Mardia & Jupp 2000): the first moment is PLV, and the second is a genuinely different quantity. Opening a second moment is opening a second channel.

We apply this qutrit to the dyad and look at where the coupling now lands.

## 2. Apply it to the Kuramoto dyad

We take the same synthetic dyad as the rest of the capstone — two coupled Kuramoto oscillators with a known coupling $K$ — but now embed each oscillator's phase trace with `multifreq_state` instead of the naive qubit. Joining the two qutrits with `joint_density_from_states` gives a $9\times 9$ density $\rho_{AB}$, one slot per pair $|i\rangle_A|j\rangle_B$ at flat index $i\cdot 3 + j$.

Two off-diagonal elements carry the cross-signal phase structure (`embeddings` module, index convention). For a phase difference $\delta = \theta_A - \theta_B$:

- $\rho_{AB}[3,1]$ — the $|1\rangle_A|0\rangle_B$ versus $|0\rangle_A|1\rangle_B$ coherence — carries the **first** circular moment, $\tfrac19\langle e^{i\delta}\rangle$. This is the same quantity (up to the prefactor) that the naive qubit held: PLV's channel.
- $\rho_{AB}[6,2]$ — the $|2\rangle_A|0\rangle_B$ versus $|0\rangle_A|2\rangle_B$ coherence — carries the **second** circular moment, $\tfrac19\langle e^{2i\delta}\rangle$. This element lives on the $|2\rangle$ level, which the naive qubit did not have. It is the new channel.

We build both densities at a coupling where the dyad is well locked, print the two qutrit coherences against the naive one, and draw $\rho_{AB}$.

In [ ]:
# Same dyad as the capstone, at a coupling where it is well locked.
K = 0.5
theta_a, theta_b = simulate_kuramoto_dyad(K=K, duration=200.0, seed=0)

# Naive 4x4 joint density (for contrast): its single coupling channel is rho[1, 2].
rho_naive = joint_density_matrix(theta_a, theta_b)

# Richer 9x9 qutrit joint density: two coupling channels, at rho[3, 1] and rho[6, 2].
psi_a = multifreq_state(theta_a, harmonics=(1, 2))
psi_b = multifreq_state(theta_b, harmonics=(1, 2))
rho_qutrit = joint_density_from_states(psi_a, psi_b)

print(f"coupling K = {K}    PLV = {plv(theta_a, theta_b):.3f}")
print()
print(f"naive   rho shape {rho_naive.shape}   single coupling channel  |rho[1,2]| = {abs(rho_naive[1, 2]):.4f}")
print(f"qutrit  rho shape {rho_qutrit.shape}")
print(f"   first-moment channel   |rho[3,1]| = {abs(rho_qutrit[3, 1]):.4f}   (PLV's channel, also in the naive qubit)")
print(f"   second-moment channel  |rho[6,2]| = {abs(rho_qutrit[6, 2]):.4f}   (NEW -- lives on the |2> level)")

**Read the output.** The naive density is $4\times 4$ with one coupling-bearing element, $|\rho[1,2]|$, whose magnitude is $\mathrm{PLV}/4$. The qutrit density is larger — $9\times 9$ — and it carries the coupling in **two** places. The first-moment channel $|\rho[3,1]|$ is nonzero, as it must be — it is the same first circular moment PLV reads, scaled by the qutrit's $\tfrac19$ prefactor. The decisive line is the last one: $|\rho[6,2]|$, the second-moment channel, is *also* nonzero. That element does not exist in the naive density at all — it lives on the $|2\rangle$ level, which the two-level qubit could not represent. The richer embedding has put coupling information somewhere new. Let us see it in the matrix itself.

In [ ]:
# Draw the richer joint density. The two coupling-bearing coherences are highlighted.
fig = viz.plot_density_matrix(
    rho_qutrit,
    title="Qutrit joint density "
    + r"$\rho_{AB}$"
    + f"  (Kuramoto dyad, K = {K})",
)
# Mark the first-moment (3, 1) and second-moment (6, 2) coherences on the Re(rho) panel.
ax_re = fig.axes[0]
for (row, col), label in [((3, 1), "1st"), ((6, 2), "2nd")]:
    for (r, c) in [(row, col), (col, row)]:
        ax_re.add_patch(
            plt.Rectangle(
                (c - 0.5, r - 0.5), 1, 1,
                fill=False, edgecolor=COLORS["highlight"], linewidth=2.5,
            )
        )
    ax_re.annotate(
        label, (col, row), xytext=(0, 14), textcoords="offset points",
        ha="center", color=COLORS["highlight"], fontsize=10, fontweight="bold",
    )
plt.show()

**Read the figure.** Two heatmaps — the real and imaginary parts of the $9\times 9$ qutrit density $\rho_{AB}$, each cell annotated with its value. Most of the structure is on the diagonal (the populations) and along the first few off-diagonals; the off-diagonal coupling content is small in absolute terms because the qutrit spreads its amplitude across three levels (the $\tfrac19$ prefactor), so do not expect vivid colour — read the highlighted cells.

Two pairs of cells are boxed in rose. The pair at $(3,1)$ and $(1,3)$ is the **first**-moment coherence: the channel the naive qubit already had, carrying $\langle e^{i\delta}\rangle$. The pair at $(6,2)$ and $(2,6)$ is the **second**-moment coherence, carrying $\langle e^{2i\delta}\rangle$. This second pair is the whole point of the enrichment: it sits on rows and columns indexed by the $|2\rangle$ level, which is absent from the $4\times 4$ naive density. The naive embedding had one coupling channel; the richer embedding has opened a second one. Whether that second channel carries anything *real* — and anything *independent* of the first — is the next two questions.

## 3. The second moment is real structure on the dyad

A new matrix element is only worth having if it carries signal. So we ask: does $|\rho_{AB}[6,2]|$ respond to the coupling, the way a genuine coupling channel should? We sweep $K$ across the locking transition and track the second-moment coherence at each value.

The expectation, from the physics of the dyad: as $K$ rises, the two oscillators lock, the phase difference $\delta$ concentrates, and *every* circular moment of a concentrated distribution grows away from zero — the second along with the first (Mardia & Jupp 2000). So $|\rho_{AB}[6,2]|$ should climb with $K$. We sweep and plot it beside the first-moment channel to see both rise.

In [ ]:
# Sweep K across the locking transition; track both qutrit coupling channels.
k_grid = np.linspace(0.0, 0.5, 12)
first_moment = np.empty_like(k_grid)
second_moment = np.empty_like(k_grid)
plv_curve = np.empty_like(k_grid)
for idx, k in enumerate(k_grid):
    t_a, t_b = simulate_kuramoto_dyad(K=float(k), duration=200.0, seed=0)
    rho = joint_density_from_states(multifreq_state(t_a), multifreq_state(t_b))
    first_moment[idx] = abs(rho[3, 1])
    second_moment[idx] = abs(rho[6, 2])
    plv_curve[idx] = plv(t_a, t_b)

print(f"|rho[6,2]| (2nd-moment channel):  low K = {second_moment[0]:.4f}    high K = {second_moment[-1]:.4f}")
print(f"|rho[3,1]| (1st-moment channel):  low K = {first_moment[0]:.4f}    high K = {first_moment[-1]:.4f}")
print(f"PLV:                              low K = {plv_curve[0]:.4f}    high K = {plv_curve[-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(
    k_grid, second_moment, "o-", color=COLORS["highlight"], lw=2.4, markersize=8,
    markeredgecolor="white", markeredgewidth=0.8,
    label=r"$|\rho_{AB}[6,2]|$  second moment  (NEW channel)",
)
ax.plot(
    k_grid, first_moment, "o-", color=COLORS["quantum"], lw=2.4, markersize=8,
    markeredgecolor="white", markeredgewidth=0.8,
    label=r"$|\rho_{AB}[3,1]|$  first moment  (PLV's channel)",
)
ax.set_xlabel("coupling strength  K")
ax.set_ylabel("coherence magnitude")
ax.set_title("The qutrit's second-moment channel grows as the dyad locks", pad=10)
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

**Read the figure.** Two rising curves against the coupling $K$. The sky-blue curve is the first-moment coherence $|\rho_{AB}[3,1]|$ — PLV's channel — climbing as the dyad locks, exactly as the naive analysis would predict. The rose curve is the new second-moment coherence $|\rho_{AB}[6,2]|$: it starts near zero at $K=0$ (uncoupled oscillators, no concentrated phase difference, so the second moment averages away) and rises with $K$, reaching a clearly nonzero plateau by the time the dyad is locked.

The rose curve is the evidence that the new channel is not an artefact. As the coupling tightens the phase difference, the second circular moment grows along with the first, and the qutrit faithfully records it in $\rho_{AB}[6,2]$. The embedding is capturing genuine higher-order phase structure of the dyad — structure the naive qubit, lacking a $|2\rangle$ level, had no place to put. Note that on *this* dyad the two curves rise together, both driven by the single knob $K$. That coincidence is exactly what the next section pulls apart.

## 4. The crucial point — the second moment is an independent degree of freedom

On the dyad of section 3 the first and second moments rose together, both pulled by the one coupling $K$. That could leave a nagging worry: perhaps the second moment is merely a function of the first — rising whenever PLV rises — in which case the new channel would be as redundant as the old one. It is not, and seeing why is the hinge of the whole fix.

The first and second circular moments, $\langle e^{i\delta}\rangle$ and $\langle e^{2i\delta}\rangle$, are **different functionals of the phase-difference distribution** (Mardia & Jupp 2000, ch. 3). On the single-knob dyad they happen to move together, but nothing forces that in general: one can hold the first moment *fixed* — hence PLV fixed — while changing the second. To show it, we step off the dyad for a moment and draw phase differences directly from a distribution whose first and second moments we set by hand, using `sample_phase_difference`, which samples $\delta$ from $p(\delta) \propto 1 + 2a_1\cos\delta + 2a_2\cos 2\delta$ with first moment $a_1$ and second moment $a_2$.

This is a **short illustration, not the experiment**. We build two small samples that share $a_1$ (so they share PLV, and hence share the first-moment channel $\rho_{AB}[3,1]$) but differ in $a_2$ (so they differ in the second-moment channel $\rho_{AB}[6,2]$). The full discriminating experiment — matched-PLV ensembles separated by a quantum measure with controls and significance — is the work of `15`. Here we only show the structural opening: a place in $\rho_{AB}$ where two equal-PLV couplings part ways.

In [ ]:
# A short teaser: two small phase-difference samples, matched 1st moment, different 2nd.
# A phase-symmetric reference theta_ref makes the marginals near-uniform, so only the
# DIFFERENCE moments survive in the joint coherences.
n = 40_000
a1 = 0.4                      # shared first moment -> shared PLV, shared rho[3,1]
theta_ref = np.linspace(0.0, 2 * np.pi, n, endpoint=False)


def qutrit_coherences(a2, *, seed):
    """Build the qutrit rho_AB for a sample with first moment a1, second moment a2."""
    delta = sample_phase_difference(n, a1=a1, a2=a2, seed=seed)
    rho = joint_density_from_states(
        multifreq_state(theta_ref), multifreq_state(theta_ref - delta)
    )
    return abs(rho[3, 1]), abs(rho[6, 2])


first_lo, second_lo = qutrit_coherences(a2=0.0, seed=7)   # no second-order structure
first_hi, second_hi = qutrit_coherences(a2=0.3, seed=7)   # nonzero second-order structure

print(f"{'sample':<22}{'a2 (set)':>10}{'|rho[3,1]| (1st)':>20}{'|rho[6,2]| (2nd)':>20}")
print("-" * 72)
print(f"{'sample 1':<22}{0.0:>10.2f}{first_lo:>20.4f}{second_lo:>20.4f}")
print(f"{'sample 2':<22}{0.3:>10.2f}{first_hi:>20.4f}{second_hi:>20.4f}")
print()
print(f"first-moment channel matched to within  {abs(first_hi - first_lo):.4f}")
print(f"second-moment channel differs by         {abs(second_hi - second_lo):.4f}")

**Read the output.** Two samples, built to share their first circular moment $a_1$ but to differ in their second, $a_2$. Read the two channels across the rows. The first-moment channel $|\rho_{AB}[3,1]|$ is the **same** for both samples, to within a few ten-thousandths — they share $a_1$, so they share PLV, so they share the channel the naive embedding would have used. A PLV-based analysis, and the naive qubit, see these two samples as identical.

The second-moment channel $|\rho_{AB}[6,2]|$ tells them apart: it is near zero for sample 1 (which has no second-order structure, $a_2 = 0$) and clearly nonzero for sample 2 ($a_2 = 0.3$). Same first moment, different second moment — and the qutrit records the difference in an element PLV cannot touch. That is the redundancy of `13` broken **in principle**: the richer $\rho_{AB}$ has a channel that distinguishes couplings PLV must call equal. The new channel is not a function of the old one; it is an independent degree of freedom.

## Your turn

**Warm-up.** In section 3, predict what the rose curve $|\rho_{AB}[6,2]|$ would look like if you replaced the qutrit `multifreq_state(theta, harmonics=(1, 2))` with the naive phase qubit. Which element of the naive $4\times 4$ density would you look in for a *second*-moment coherence — and what goes wrong when you try to point at one? Tie your answer to the number of levels each embedding has.

**Go further.** The teaser in section 4 fixed the first moment $a_1$ and varied the second $a_2$. Suppose instead you fixed *both* the first and second moments and wanted two samples that differ only in the **third** circular moment $\langle e^{3i\delta}\rangle$. Describe in words which channel of $\rho_{AB}$ would then have to carry the distinction, and what you would change in the `multifreq_state` call so the joint density has a slot for it. Why would the qutrit of this notebook be blind to such a pair?

**Challenge.** Argue, in words, why a nonzero gap in $|\rho_{AB}[6,2]|$ between two matched-PLV samples is a *necessary* condition for a quantum coupling measure to separate them, but explain what extra step section 4 has *not* yet taken to make it a *sufficient* demonstration. What would you still need to compute on the two samples — and across many such pairs — before you could claim a quantum measure beats PLV? (You are describing the experiment `15` runs; you are not running it here.)

## What you built

- You took the diagnosis of `13` — that the redundancy was the *naive embedding's* fault, not quantum OT's — and built the fix: you applied the **multi-frequency qutrit** to the Kuramoto dyad and assembled its $9\times 9$ joint density $\rho_{AB}$.
- You located the **two** coupling-bearing coherences and named what each holds: $\rho_{AB}[3,1]$ carries the first circular moment (PLV's channel, the only one the naive qubit had), and $\rho_{AB}[6,2]$ carries the **second** moment — a channel that lives on the $|2\rangle$ level the two-level qubit did not possess.
- You showed $\rho_{AB}[6,2]$ is genuine structure on the dyad: it climbs from near zero toward a clear plateau as $K$ tightens the locking, capturing higher-order phase structure the naive embedding had no place to record.
- You showed, with a short two-sample illustration, that the second moment is an **independent** degree of freedom: two samples with matched PLV (matched $\rho_{AB}[3,1]$) can differ in $\rho_{AB}[6,2]$. The redundancy is broken in principle — the richer $\rho_{AB}$ encodes more than $|\mathrm{PLV}|$.

You have built a better tool. Where the naive embedding fed the quantum measure a single number that *was* PLV, the qutrit hands it a state with a second, independent channel — and you have seen that channel separate two couplings PLV must call identical. In `15_discriminating_experiment` you put this channel to work: you build whole ensembles of matched-PLV dyads that QOT can separate but PLV cannot, turning today's structural opening into a measured advantage.

## References

- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C — the phase-locking value, $\mathrm{PLV} = |\langle e^{i\Delta\theta}\rangle|$, the first circular moment carried by $\rho_{AB}[3,1]$.
- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular (trigonometric) moments of a phase distribution; the first and second moments are distinct functionals, the basis for the independent channel opened here.
- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091 — quantum-transport coupling measures that read the enriched $\rho_{AB}$.

**Previous:** `notebooks/04_QuantumOptimalTransport/13_the_redundancy_reveal.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/15_discriminating_experiment.ipynb`